In [0]:
# Read the existing Bronze Delta table
df_bronze = spark.table(
    "workspace.greenhouse_bronze.raw_production"
)

# Inspect column names and types
df_bronze.printSchema()

# Display a sample
display(df_bronze.limit(5))

print("Bronze records:", df_bronze.count())

root
 |-- REF_DATE: string (nullable = true)
 |-- GEO: string (nullable = true)
 |-- DGUID: string (nullable = true)
 |-- Commodity: string (nullable = true)
 |-- Production and value: string (nullable = true)
 |-- Estimates: string (nullable = true)
 |-- UOM: string (nullable = true)
 |-- UOM_ID: string (nullable = true)
 |-- SCALAR_FACTOR: string (nullable = true)
 |-- SCALAR_ID: string (nullable = true)
 |-- VECTOR: string (nullable = true)
 |-- COORDINATE: string (nullable = true)
 |-- VALUE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- SYMBOL: string (nullable = true)
 |-- TERMINATED: string (nullable = true)
 |-- DECIMALS: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



REF_DATE,GEO,DGUID,Commodity,Production and value,Estimates,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS,ingestion_timestamp,source_file
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Production,Pounds,Pounds,254,units,0,v47951,1.1.1.1,3547068,null,null,null,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Average price,Dollars per pound,Dollars per pound,119,units,0,v47963,1.1.2.2,0.22,null,null,null,2,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Farm gate value,Dollars,Dollars,81,units,0,v47975,1.1.3.3,783450,null,null,null,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Newfoundland and Labrador and Prince Edward Island,null,Fresh tomatoes [11422151],Production,Pounds,Pounds,254,units,0,v47953,4.1.1.1,9300,null,null,t,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Newfoundland and Labrador and Prince Edward Island,null,Fresh tomatoes [11422151],Average price,Dollars per pound,Dollars per pound,119,units,0,v47965,4.1.2.2,0.34,null,null,t,2,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv


Bronze records: 16684


In [0]:
import re

# Convert source column names to snake_case
def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

# Apply standardized column names
df_silver = df_bronze

for old_name in df_bronze.columns:
    new_name = clean_column_name(old_name)
    df_silver = df_silver.withColumnRenamed(
        old_name, new_name
    )

# Check the transformed schema
df_silver.printSchema()

# Preview the transformed data
display(df_silver.limit(5))

print("Silver working records:", df_silver.count())


root
 |-- ref_date: string (nullable = true)
 |-- geo: string (nullable = true)
 |-- dguid: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- production_and_value: string (nullable = true)
 |-- estimates: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- uom_id: string (nullable = true)
 |-- scalar_factor: string (nullable = true)
 |-- scalar_id: string (nullable = true)
 |-- vector: string (nullable = true)
 |-- coordinate: string (nullable = true)
 |-- value: string (nullable = true)
 |-- status: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- terminated: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



ref_date,geo,dguid,commodity,production_and_value,estimates,uom,uom_id,scalar_factor,scalar_id,vector,coordinate,value,status,symbol,terminated,decimals,ingestion_timestamp,source_file
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Production,Pounds,Pounds,254,units,0,v47951,1.1.1.1,3547068,null,null,null,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Average price,Dollars per pound,Dollars per pound,119,units,0,v47963,1.1.2.2,0.22,null,null,null,2,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Canada,2016A000011124,Fresh tomatoes [11422151],Farm gate value,Dollars,Dollars,81,units,0,v47975,1.1.3.3,783450,null,null,null,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Newfoundland and Labrador and Prince Edward Island,null,Fresh tomatoes [11422151],Production,Pounds,Pounds,254,units,0,v47953,4.1.1.1,9300,null,null,t,0,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv
1955,Newfoundland and Labrador and Prince Edward Island,null,Fresh tomatoes [11422151],Average price,Dollars per pound,Dollars per pound,119,units,0,v47965,4.1.2.2,0.34,null,null,t,2,2026-10-09T20:17:26.754Z,/Volumes/workspace/greenhouse_bronze/raw_files/32100456.csv


Silver working records: 16684


In [0]:
from pyspark.sql.functions import col

# Convert columns to appropriate data types
df_silver = (
    df_silver
    .withColumn("ref_date", col("ref_date").cast("int"))
    .withColumn("value", col("value").cast("double"))
)

# Verify data types
df_silver.printSchema()

# Examine status codes
display(
    df_silver.groupBy("status")
    .count()
    .orderBy("status")
)

# Check missing values and total records
print("Null measurement values:",
      df_silver.filter(col("value").isNull()).count())

print("Silver working records:", df_silver.count())

root
 |-- ref_date: integer (nullable = true)
 |-- geo: string (nullable = true)
 |-- dguid: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- production_and_value: string (nullable = true)
 |-- estimates: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- uom_id: string (nullable = true)
 |-- scalar_factor: string (nullable = true)
 |-- scalar_id: string (nullable = true)
 |-- vector: string (nullable = true)
 |-- coordinate: string (nullable = true)
 |-- value: double (nullable = true)
 |-- status: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- terminated: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



status,count
null,8987
..,2896
E,740
F,511
x,3550


Null measurement values: 6957
Silver working records: 16684


In [0]:

from pyspark.sql.functions import col, count, sum, when

# Compare status codes with missing measurements
df_null_analysis = (
    df_silver
    .groupBy("status")
    .agg(
        count("*").alias("total_records"),
        sum(
            when(col("value").isNull(), 1).otherwise(0)
        ).alias("null_values"),
        sum(
            when(col("value").isNotNull(), 1).otherwise(0)
        ).alias("non_null_values")
    )
    .orderBy(col("null_values").desc())
)

display(df_null_analysis)



status,total_records,null_values,non_null_values
x,3550,3550,0
..,2896,2896,0
F,511,511,0
null,8987,0,8987
E,740,0,740


In [0]:

from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "measurement_quality",
    when(col("status") == "x", "CONFIDENTIAL")
    .when(col("status") == "..", "NOT_AVAILABLE")
    .when(col("status") == "F", "UNRELIABLE")
    .when(col("status") == "E", "USE_WITH_CAUTION")
    .when(col("value").isNotNull(), "AVAILABLE")
    .otherwise("REVIEW_REQUIRED")
)

# Check the new classification
display(
    df_silver.groupBy("measurement_quality")
    .count()
    .orderBy("measurement_quality")
)

print("Silver working records:", df_silver.count())


measurement_quality,count
AVAILABLE,8987
CONFIDENTIAL,3550
NOT_AVAILABLE,2896
UNRELIABLE,511
USE_WITH_CAUTION,740


Silver working records: 16684


In [0]:

from pyspark.sql.functions import col

# Identify the original source columns
source_columns = [
    c for c in df_bronze.columns
    if c not in ["ingestion_timestamp", "source_file"]
]

# Count distinct original source records
distinct_records = (
    df_bronze
    .select(*[col(c) for c in source_columns])
    .distinct()
    .count()
)

total_records = df_bronze.count()

print("Total Bronze records:", total_records)
print("Distinct source records:", distinct_records)
print("Exact duplicate records:", total_records - distinct_records)


Total Bronze records: 16684
Distinct source records: 16684
Exact duplicate records: 0


In [0]:

from pyspark.sql.functions import col, trim, expr

# Load the Bronze Delta table again
df_bronze = spark.table(
    "workspace.greenhouse_bronze.raw_production"
)

df_cast_check = (
    df_bronze
    .withColumn(
        "parsed_value",
        expr("try_cast(VALUE as double)")
    )
    .withColumn(
        "parsed_year",
        expr("try_cast(REF_DATE as int)")
    )
)

# Find non-empty measurements that cannot be converted
invalid_values = df_cast_check.filter(
    col("VALUE").isNotNull()
    & (trim(col("VALUE")) != "")
    & col("parsed_value").isNull()
).count()

# Find non-empty years that cannot be converted
invalid_years = df_cast_check.filter(
    col("REF_DATE").isNotNull()
    & (trim(col("REF_DATE")) != "")
    & col("parsed_year").isNull()
).count()

print("Invalid numeric measurements:", invalid_values)
print("Invalid reporting years:", invalid_years)


Invalid numeric measurements: 0
Invalid reporting years: 0


In [0]:

from pyspark.sql.functions import col, count

# Read Bronze directly so this cell works independently
df_key_check = spark.table(
    "workspace.greenhouse_bronze.raw_production"
)

# Check whether year + vector identifies a unique observation
duplicate_keys = (
    df_key_check
    .groupBy("REF_DATE", "VECTOR")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

print("Duplicate year-vector combinations:", duplicate_keys.count())

display(duplicate_keys.limit(10))


Duplicate year-vector combinations: 0


REF_DATE,VECTOR,record_count


In [0]:

import re
from pyspark.sql.functions import col, when, expr

# 1. Read persisted Bronze data
df_silver = spark.table(
    "workspace.greenhouse_bronze.raw_production"
)

# 2. Standardize column names
def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

for old_name in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(
        old_name, clean_column_name(old_name)
    )

# 3. Safely convert data types
df_silver = (
    df_silver
    .withColumn("ref_date", expr("try_cast(ref_date as int)"))
    .withColumn("value", expr("try_cast(value as double)"))
)

# 4. Classify measurement quality
df_silver = df_silver.withColumn(
    "measurement_quality",
    when(col("status") == "x", "CONFIDENTIAL")
    .when(col("status") == "..", "NOT_AVAILABLE")
    .when(col("status") == "F", "UNRELIABLE")
    .when(col("status") == "E", "USE_WITH_CAUTION")
    .when(col("value").isNotNull(), "AVAILABLE")
    .otherwise("REVIEW_REQUIRED")
)

# 5. Save Silver Delta table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.greenhouse_silver.production_clean")
)

# 6. Verify saved records
silver_count = spark.table(
    "workspace.greenhouse_silver.production_clean"
).count()

print("Silver Delta records:", silver_count)
print("Counts match:", silver_count == 16684)


Silver Delta records: 16684
Counts match: True


In [0]:

from pyspark.sql.functions import col

# Read the persisted Silver table
df_saved = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

# Verify the schema
df_saved.printSchema()

# Check measurement quality classifications
display(
    df_saved.groupBy("measurement_quality")
    .count()
    .orderBy("measurement_quality")
)

# Check for unclassified records
review_count = df_saved.filter(
    col("measurement_quality") == "REVIEW_REQUIRED"
).count()

print("Records requiring review:", review_count)


root
 |-- ref_date: integer (nullable = true)
 |-- geo: string (nullable = true)
 |-- dguid: string (nullable = true)
 |-- commodity: string (nullable = true)
 |-- production_and_value: string (nullable = true)
 |-- estimates: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- uom_id: string (nullable = true)
 |-- scalar_factor: string (nullable = true)
 |-- scalar_id: string (nullable = true)
 |-- vector: string (nullable = true)
 |-- coordinate: string (nullable = true)
 |-- value: double (nullable = true)
 |-- status: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- terminated: string (nullable = true)
 |-- decimals: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- measurement_quality: string (nullable = true)



measurement_quality,count
AVAILABLE,8987
CONFIDENTIAL,3550
NOT_AVAILABLE,2896
UNRELIABLE,511
USE_WITH_CAUTION,740


Records requiring review: 0
